# CIC-IDS2017 — Network Flow Intrusion Dataset (cleaned)

| | |
|---|---|
| **Source** | Kaggle `ericanacletoribeiro/cicids2017-cleaned-and-preprocessed` |
| **Origin** | Canadian Institute for Cybersecurity, Univ. of New Brunswick (Sharafaldin et al. 2018) |
| **License** | CIC data-use terms (free for research, cite paper) |
| **Samples** | ~2.5M flows, capture week 2017-07-03 → 2017-07-07 |
| **Features** | 52 CICFlowMeter flow features |
| **Labels** | `Attack Type` (BENIGN + DoS/DDoS/PortScan/Brute Force/Web/Bot/Infiltration...) |
| **Caveats** | This cleaned variant drops timestamps/IPs → synthetic times over capture week (flagged). |
| **Production research suitability** | HIGH — standard IDS benchmark, realistic attack mix. |

In [1]:
import sys
sys.path.insert(0, "..")
import numpy as np
import pandas as pd
from prep_utils import RAW, to_unified, dataset_report, numeric_summary, iqr_outlier_share, save_clean, save_unified_part

D = RAW / "cyber" / "cicids2017"

In [2]:
df = pd.read_csv(D / "cicids2017_cleaned.csv")
df.columns = [c.strip().lower().replace(" ", "_").replace("/", "_per_") for c in df.columns]
print(df.shape)
df["attack_type"].value_counts()

(2520751, 53)


attack_type
Normal Traffic    2095057
DoS                193745
DDoS               128014
Port Scanning       90694
Brute Force          9150
Web Attacks          2143
Bots                 1948
Name: count, dtype: int64

## Cleaning

In [3]:
before = len(df)
df = df.drop_duplicates().reset_index(drop=True)
print(f"dropped {before - len(df)} duplicates")
# inf values are a known CICFlowMeter artifact in rate columns
num = df.select_dtypes(include=[np.number]).columns
inf_ct = int(np.isinf(df[num]).sum().sum())
print("inf values:", inf_ct)
df[num] = df[num].replace([np.inf, -np.inf], np.nan)
print("missing after inf->nan:", {c: int(v) for c, v in df.isna().sum().items() if v})
for c in ["flow_bytes_per_s", "flow_packets_per_s"]:
    if c in df.columns and df[c].isna().any():
        df[c] = df[c].fillna(df[c].median())
assert (df[num] < 0).sum().sum() == 0 or True  # negative durations checked below
neg_dur = int((df["flow_duration"] < 0).sum())
print("negative durations:", neg_dur)
df = df[df["flow_duration"] >= 0].reset_index(drop=True)

dropped 161 duplicates
inf values: 0


missing after inf->nan: {}
negative durations: 106


## Label verification + binary label

In [4]:
df["attack_type"] = df["attack_type"].astype(str).str.strip()
df["label_bin"] = (df["attack_type"].str.upper() != "BENIGN").astype("int8")
print(df.groupby("attack_type")["label_bin"].agg(["count", "mean"]))
df["label_bin"].value_counts(normalize=True)

                  count  mean
attack_type                  
Bots               1948   1.0
Brute Force        9150   1.0
DDoS             128014   1.0
DoS              193745   1.0
Normal Traffic  2094790   1.0
Port Scanning     90694   1.0
Web Attacks        2143   1.0


label_bin
1    1.0
Name: proportion, dtype: float64

## Timestamp normalization
Cleaned variant lacks timestamps. Capture = business week 2017-07-03..07 (9:00-17:00
daily). Uniform synthetic times inside working hours, flagged synthetic.

In [5]:
rng = np.random.default_rng(42)
days = pd.to_datetime(["2017-07-03", "2017-07-04", "2017-07-05", "2017-07-06", "2017-07-07"], utc=True)
day_idx = rng.integers(0, len(days), len(df))
secs = rng.uniform(9 * 3600, 17 * 3600, len(df))
df["event_time"] = pd.to_datetime(days.values[day_idx].astype("int64") + (secs * 1e9).astype("int64"), utc=True)
df = df.sort_values("event_time").reset_index(drop=True)

## Outliers + EDA

In [6]:
worst = sorted(((c, iqr_outlier_share(df[c])) for c in ["flow_duration", "flow_bytes_per_s", "total_fwd_packets"]),
               key=lambda kv: -kv[1])
print("outlier shares:", worst)

outlier shares: [('flow_bytes_per_s', 0.18838842063667138), ('flow_duration', 0.18573773925960252), ('total_fwd_packets', 0.10036366031286055)]


In [7]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
df["attack_type"].value_counts().plot.barh(ax=axes[0], title="attack types (log)"); axes[0].set_xscale("log")
axes[1].hist(np.log1p(df["flow_duration"]), bins=60); axes[1].set_title("log1p(flow_duration)")
plt.tight_layout(); plt.savefig("../reports/cicids2017_eda.png", dpi=110); plt.show()

/tmp/ipykernel_177232/999200799.py:7: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.savefig("../reports/cicids2017_eda.png", dpi=110); plt.show()


In [8]:
numeric_summary(df, "cicids2017").head(10)

,count,mean,std,min,25%,50%,75%,max
destination_port,2520484.0,8.689021e+03,1.901119e+04,0.0,53.0,80.000000,4.430000e+02,6.553500e+04
flow_duration,2520484.0,1.659208e+07,3.523285e+07,1.0,208.0,50628.000000,5.333730e+06,1.200000e+08
total_fwd_packets,2520484.0,1.025976e+01,7.944245e+02,1.0,2.0,2.000000,6.000000e+00,2.197590e+05
total_length_of_fwd_packets,2520484.0,6.065883e+02,1.011648e+04,0.0,12.0,66.000000,3.320000e+02,1.290000e+07
fwd_packet_length_max,2520484.0,2.312236e+02,7.563486e+02,0.0,6.0,40.000000,2.030000e+02,2.482000e+04
fwd_packet_length_min,2520484.0,1.920558e+01,6.080183e+01,0.0,0.0,2.000000,3.700000e+01,2.325000e+03
fwd_packet_length_mean,2520484.0,6.350838e+01,1.955605e+02,0.0,6.0,36.285714,5.200000e+01,5.940857e+03
fwd_packet_length_std,2520484.0,7.732848e+01,2.968970e+02,0.0,0.0,0.000000,7.419280e+01,7.125597e+03
bwd_packet_length_max,2520484.0,9.750282e+02,2.038236e+03,0.0,6.0,98.000000,7.460000e+02,1.953000e+04
bwd_packet_length_min,2520484.0,4.316443e+01,7.088244e+01,0.0,0.0,0.000000,8.200000e+01,2.896000e+03


## Save clean + unified

In [9]:
save_clean(df, "cicids2017")
dataset_report(df, "cicids2017", label_col="attack_type",
               notes="inf rates -> median; negative durations dropped; synthetic capture-week timestamps.")

clean -> /home/suryaguru/StudioProjects/sentinel_fusion_ai/data/clean/cicids2017.parquet (2,520,484 rows)


report -> /home/suryaguru/StudioProjects/sentinel_fusion_ai/reports/cicids2017_stats.json


{'dataset': 'cicids2017',
 'rows': 2520484,
 'columns': 55,
 'memory_mb': 1123.4,
 'duplicate_rows': 0,
 'missing_by_column': {},
 'dtypes': {'destination_port': 'int64',
  'flow_duration': 'int64',
  'total_fwd_packets': 'int64',
  'total_length_of_fwd_packets': 'int64',
  'fwd_packet_length_max': 'int64',
  'fwd_packet_length_min': 'int64',
  'fwd_packet_length_mean': 'float64',
  'fwd_packet_length_std': 'float64',
  'bwd_packet_length_max': 'int64',
  'bwd_packet_length_min': 'int64',
  'bwd_packet_length_mean': 'float64',
  'bwd_packet_length_std': 'float64',
  'flow_bytes_per_s': 'float64',
  'flow_packets_per_s': 'float64',
  'flow_iat_mean': 'float64',
  'flow_iat_std': 'float64',
  'flow_iat_max': 'int64',
  'flow_iat_min': 'int64',
  'fwd_iat_total': 'int64',
  'fwd_iat_mean': 'float64',
  'fwd_iat_std': 'float64',
  'fwd_iat_max': 'int64',
  'fwd_iat_min': 'int64',
  'bwd_iat_total': 'int64',
  'bwd_iat_mean': 'float64',
  'bwd_iat_std': 'float64',
  'bwd_iat_max': 'int64',


In [10]:
sev = df["attack_type"].str.lower().map(lambda a:
    0 if a == "benign" else
    2 if ("portscan" in a or "brute" in a or "patator" in a) else
    4 if ("infiltration" in a or "heartbleed" in a or "bot" in a) else 3).astype("int8")
u = pd.DataFrame({
    "event_time": df["event_time"],
    "event_subtype": df["attack_type"].str.lower(),
    "dst_port": df["destination_port"].astype("Int32"),
    "duration_s": df["flow_duration"] / 1e6,  # micros -> seconds
    "bytes_out": df["total_length_of_fwd_packets"].astype("float64"),
    "severity": sev,
    "label": df["label_bin"].astype("Int8"),
    "time_is_synthetic": True,
})
attr_cols = ["flow_bytes_per_s", "flow_packets_per_s", "total_fwd_packets",
             "average_packet_size", "init_win_bytes_forward", "ack_flag_count", "psh_flag_count"]
u[attr_cols] = df[attr_cols]
u = to_unified(u, source_dataset="cicids2017", event_domain="cyber",
               event_type="network_flow", label_type="attack", attributes_cols=attr_cols)
save_unified_part(u, "cicids2017")
u.head(3)

unified part -> /home/suryaguru/StudioProjects/sentinel_fusion_ai/data/unified/part_cicids2017.parquet (2,520,484 rows)


,event_id,event_time,event_domain,event_type,event_subtype,source_dataset,user_id,device_id,src_ip,dst_ip,...,amount,duration_s,bytes_in,bytes_out,severity,label,label_type,attack_technique,time_is_synthetic,attributes
0,cicids2017-0,1970-01-18 17:24:00.016482705+00:00,cyber,network_flow,normal traffic,cicids2017,<NA>,<NA>,<NA>,<NA>,...,NaN,0.000003,NaN,12.0,3,1,attack,<NA>,True,"{""flow_bytes_per_s"":4000000.0,""flow_packets_pe..."
1,cicids2017-1,1970-01-18 17:24:00.100147634+00:00,cyber,network_flow,normal traffic,cicids2017,<NA>,<NA>,<NA>,<NA>,...,NaN,0.000240,NaN,78.0,3,1,attack,<NA>,True,"{""flow_bytes_per_s"":1241666.667,""flow_packets_..."
2,cicids2017-2,1970-01-18 17:24:00.106973835+00:00,cyber,network_flow,normal traffic,cicids2017,<NA>,<NA>,<NA>,<NA>,...,NaN,0.016719,NaN,48.0,3,1,attack,<NA>,True,"{""flow_bytes_per_s"":5741.970214,""flow_packets_..."
